<a href="https://colab.research.google.com/github/sabihadudhia/Thesis-Hallucination-Benchmarks/blob/main/Phi_4_HaluEval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess, os, re, json, time, random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import notebook_login
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
torch.manual_seed(42)

notebook_login()

In [2]:
subprocess.run(["git", "clone", "https://github.com/RUCAIBox/HaluEval.git", "HaluEval"])

CompletedProcess(args=['git', 'clone', 'https://github.com/RUCAIBox/HaluEval.git', 'HaluEval'], returncode=0)

In [3]:
with open('HaluEval/data/qa_data.json') as f:
    all_samples = [json.loads(line) for line in f]

random.seed(42)
sampled = random.sample(all_samples, 300)

instances = []
for s in sampled:
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["right_answer"], "is_hallucinated_gt": False})
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["hallucinated_answer"], "is_hallucinated_gt": True})

print(f"Total instances: {len(instances)}")

Total instances: 600


In [4]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.5 MB/s eta 0:00:00


In [5]:
model_name = "microsoft/phi-4"
tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.25M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [6]:
def normalize_response(text):
    return text.strip()

def score_halueval_instance(instance, model, tokenizer):
    prompt = (
        f"Knowledge: {instance['knowledge']}\n"
        f"Question: {instance['question']}\n"
        f"Answer: {instance['answer']}\n"
        f"Is the answer hallucinated? Answer Yes or No only."
    )

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    response = normalize_response(response).lower()

    if "yes" in response[:10]:
        predicted = True
    elif "no" in response[:10]:
        predicted = False
    else:
        return {"correct": None, "raw_response": response, "reason": "unparseable"}

    return {"correct": predicted == instance["is_hallucinated_gt"], "raw_response": response, "reason": None}

In [7]:
for i in range(10):
    result = score_halueval_instance(instances[i], model, tokenizer)
    print(f"Instance {i}: {result}")

Instance 0: {'correct': True, 'raw_response': 'no', 'reason': None}
Instance 1: {'correct': True, 'raw_response': 'yes. the answer is hallucinated because it incorrectly', 'reason': None}
Instance 2: {'correct': True, 'raw_response': 'no.', 'reason': None}
Instance 3: {'correct': False, 'raw_response': 'no. the answer is not hallucinated. roberto', 'reason': None}
Instance 4: {'correct': True, 'raw_response': 'no', 'reason': None}
Instance 5: {'correct': True, 'raw_response': 'yes.', 'reason': None}
Instance 6: {'correct': True, 'raw_response': 'no.', 'reason': None}
Instance 7: {'correct': True, 'raw_response': 'yes. the correct birth year for wilhelm keitel', 'reason': None}
Instance 8: {'correct': True, 'raw_response': 'no.', 'reason': None}
Instance 9: {'correct': False, 'raw_response': 'no.', 'reason': None}


In [10]:
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

save_path = '/content/drive/MyDrive/thesis_results/phi4_halueval_qa_results.jsonl'
print(f"Run started: {datetime.now().isoformat()}")

results = []
start_time = time.time()

with open(save_path, 'w') as f:
    for i, instance in enumerate(instances):
        result = score_halueval_instance(instance, model, tokenizer)
        result["instance_id"] = i
        results.append(result)
        f.write(json.dumps(result) + "\n")
        f.flush()
        os.fsync(f.fileno())   # <-- forces the write to actually reach Drive, not just OS buffer
        if i % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Progress: {i}/{len(instances)} | Elapsed: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.1f}s ({total_time/60:.1f} min)")

valid_results = [r for r in results if r["correct"] is not None]
accuracy = sum(r["correct"] for r in valid_results) / len(valid_results)
invalid_count = len(results) - len(valid_results)

print(f"Accuracy: {accuracy:.3f}")
print(f"Invalid/unparseable responses: {invalid_count} / {len(results)}")

Mounted at /content/drive
Run started: 2026-09-17T11:41:25.219880
Progress: 0/600 | Elapsed: 0.4s
Progress: 50/600 | Elapsed: 32.8s
Progress: 100/600 | Elapsed: 66.2s
Progress: 150/600 | Elapsed: 96.7s
Progress: 200/600 | Elapsed: 128.7s
Progress: 250/600 | Elapsed: 165.0s
Progress: 300/600 | Elapsed: 194.1s
Progress: 350/600 | Elapsed: 225.9s
Progress: 400/600 | Elapsed: 262.6s
Progress: 450/600 | Elapsed: 292.8s
Progress: 500/600 | Elapsed: 325.1s
Progress: 550/600 | Elapsed: 359.3s

Done. Total time: 391.8s (6.5 min)
Accuracy: 0.672
Invalid/unparseable responses: 0 / 600
